# BUSM131 — Telco Customer Churn: Kaplan-Meier Survival Analysis

Kaplan-Meier non-parametric survival analysis.

**Contents:** Overall KM curve, KM by segments (contract, internet, payment, senior, security, tech support, partner, dependents), median survival, Log-Rank tests.


In [ ]:
# Run this first if lifelines is not installed:
# !pip install lifelines

%matplotlib inline


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12


## 1. DATA LOADING & SURVIVAL SETUP


In [ ]:
print("=" * 65)


In [ ]:
print("=" * 65)

# *** UPDATE THIS PATH to your local file location ***
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Survival analysis variables:
#   T = tenure (duration in months)
#   E = event indicator (1 = churned, 0 = censored / still active)
df['T'] = df['tenure']
df['E'] = (df['Churn'] == 'Yes').astype(int)

# Handle tenure=0 (set to 0.5 to avoid zero-duration issues)
df.loc[df['T'] == 0, 'T'] = 0.5

print(f"Total customers:     {len(df)}")
print(f"Churned (event=1):   {df['E'].sum()} ({df['E'].mean()*100:.1f}%)")
print(f"Censored (event=0):  {(1-df['E']).sum()} ({(1-df['E']).mean()*100:.1f}%)")
print(f"Observation window:  0 – {df['T'].max():.0f} months")


## 2. OVERALL KAPLAN-MEIER CURVE


In [ ]:
print("\n" + "-" * 65)


In [ ]:
print("-" * 65)

kmf = KaplanMeierFitter()
kmf.fit(durations=df['T'], event_observed=df['E'], label='All Customers')

print(f"\n  Median survival time: {kmf.median_survival_time_:.1f} months")
print(f"  P(survive > 6 months):  {kmf.predict(6):.3f}")
print(f"  P(survive > 12 months): {kmf.predict(12):.3f}")
print(f"  P(survive > 24 months): {kmf.predict(24):.3f}")
print(f"  P(survive > 48 months): {kmf.predict(48):.3f}")
print(f"  P(survive > 60 months): {kmf.predict(60):.3f}")

fig, ax = plt.subplots(figsize=(10, 7))
kmf.plot_survival_function(ax=ax, ci_show=True, color='#2c3e50', linewidth=2.5)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='50% survival')
ax.set_title('Overall Kaplan-Meier Survival Curve\nTelco Customer Retention',
             fontsize=16, fontweight='bold')
ax.set_xlabel('Tenure (Months)', fontsize=13)
ax.set_ylabel('Survival Probability (Retention)', fontsize=13)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 75)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()


## 3. KM CURVES BY SEGMENTS


In [ ]:
print("\n" + "-" * 65)


### 3. KAPLAN-MEIER CURVES BY KEY SEGMENTS


In [ ]:
print("-" * 65)


In [ ]:
def plot_km_by_group(df, group_col, colors, figname, title):
    """Plot KM curves for each group, print median survival times."""
    fig, ax = plt.subplots(figsize=(10, 7))
    medians = {}
    
    for group, color in colors.items():
        mask = df[group_col] == group
        kmf_g = KaplanMeierFitter()
        kmf_g.fit(df.loc[mask, 'T'], df.loc[mask, 'E'], label=str(group))
        kmf_g.plot_survival_function(ax=ax, ci_show=True, color=color, linewidth=2)
        medians[group] = kmf_g.median_survival_time_
    
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.set_xlabel('Tenure (Months)', fontsize=13)
    ax.set_ylabel('Survival Probability', fontsize=13)
    ax.legend(fontsize=11, title=group_col)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 75)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()
    
    print(f"\n  {group_col} — Median Survival Times:")
    for g, m in medians.items():
        print(f"    {g}: {m:.1f} months")
    print(f"  Saved {figname}")


In [ ]:
# --- By Contract Type ---
plot_km_by_group(df, 'Contract',
    {'Month-to-month': '#e74c3c', 'One year': '#f39c12', 'Two year': '#2ecc71'},
    'fig13_km_contract.png',
    'Kaplan-Meier Survival Curves by Contract Type')


In [ ]:
# --- By Internet Service ---
plot_km_by_group(df, 'InternetService',
    {'Fiber optic': '#e74c3c', 'DSL': '#3498db', 'No': '#2ecc71'},
    'fig14_km_internet.png',
    'Kaplan-Meier Survival Curves by Internet Service')


In [ ]:
# --- By Payment Method ---
plot_km_by_group(df, 'PaymentMethod',
    {'Electronic check': '#e74c3c', 'Mailed check': '#f39c12',
     'Bank transfer (automatic)': '#3498db', 'Credit card (automatic)': '#2ecc71'},
    'fig15_km_payment.png',
    'Kaplan-Meier Survival Curves by Payment Method')


In [ ]:
# --- By Senior Citizen ---
df['SeniorLabel'] = df['SeniorCitizen'].map({0: 'Non-Senior', 1: 'Senior'})
plot_km_by_group(df, 'SeniorLabel',
    {'Non-Senior': '#3498db', 'Senior': '#e74c3c'},
    'fig16_km_senior.png',
    'Kaplan-Meier Survival Curves by Senior Citizen Status')


In [ ]:
# --- By Online Security ---
plot_km_by_group(df, 'OnlineSecurity',
    {'No': '#e74c3c', 'Yes': '#2ecc71', 'No internet service': '#95a5a6'},
    'fig17_km_online_security.png',
    'Kaplan-Meier Survival Curves by Online Security')


In [ ]:
# --- By Tech Support ---
plot_km_by_group(df, 'TechSupport',
    {'No': '#e74c3c', 'Yes': '#2ecc71', 'No internet service': '#95a5a6'},
    'fig18_km_tech_support.png',
    'Kaplan-Meier Survival Curves by Tech Support')


In [ ]:
# --- By Partner ---
plot_km_by_group(df, 'Partner',
    {'No': '#e74c3c', 'Yes': '#2ecc71'},
    'fig19_km_partner.png',
    'Kaplan-Meier Survival Curves by Partner Status')


In [ ]:
# --- By Dependents ---
plot_km_by_group(df, 'Dependents',
    {'No': '#e74c3c', 'Yes': '#2ecc71'},
    'fig20_km_dependents.png',
    'Kaplan-Meier Survival Curves by Dependents')


## 4. LOG-RANK TESTS


In [ ]:
print("\n" + "-" * 65)


### 4. LOG-RANK TESTS — Statistical Significance


In [ ]:
print("-" * 65)

tests = {}

# Multivariate log-rank tests
for col in ['Contract', 'InternetService', 'PaymentMethod']:
    result = multivariate_logrank_test(df['T'], df[col], df['E'])
    sig = '***' if result.p_value < 0.001 else ('**' if result.p_value < 0.01 else
          ('*' if result.p_value < 0.05 else 'n.s.'))
    tests[col] = {'chi2': result.test_statistic, 'p': result.p_value, 'sig': sig}
    print(f"  {col:25s}  χ²={result.test_statistic:>8.2f}  p={result.p_value:.2e}  {sig}")

# Pairwise log-rank tests
pairwise = [
    ('SeniorCitizen', 0, 1, 'Non-Senior vs Senior'),
    ('Partner', 'No', 'Yes', 'No Partner vs Partner'),
    ('Dependents', 'No', 'Yes', 'No Dependents vs Dependents'),
    ('gender', 'Male', 'Female', 'Male vs Female'),
    ('PaperlessBilling', 'No', 'Yes', 'Paper vs Paperless'),
]

for col, g1, g2, label in pairwise:
    m1, m2 = df[col] == g1, df[col] == g2
    result = logrank_test(df.loc[m1,'T'], df.loc[m2,'T'],
                          df.loc[m1,'E'], df.loc[m2,'E'])
    sig = '***' if result.p_value < 0.001 else ('**' if result.p_value < 0.01 else
          ('*' if result.p_value < 0.05 else 'n.s.'))
    tests[label] = {'chi2': result.test_statistic, 'p': result.p_value, 'sig': sig}
    print(f"  {label:25s}  χ²={result.test_statistic:>8.2f}  p={result.p_value:.2e}  {sig}")

# Save log-rank results table
lr_df = pd.DataFrame(tests).T
lr_df.to_csv('logrank_results.csv')


## 5. SUMMARY TABLE — MEDIAN SURVIVAL BY SEGMENT


In [ ]:
print("\n" + "-" * 65)


### 5. MEDIAN SURVIVAL TIME SUMMARY TABLE


In [ ]:
print("-" * 65)

summary_data = []
for col in ['Contract', 'InternetService', 'PaymentMethod',
            'SeniorLabel', 'OnlineSecurity', 'TechSupport', 'Partner']:
    for group in df[col].unique():
        mask = df[col] == group
        kmf_g = KaplanMeierFitter()
        kmf_g.fit(df.loc[mask, 'T'], df.loc[mask, 'E'])
        summary_data.append({
            'Variable': col,
            'Group': group,
            'n': mask.sum(),
            'Events': df.loc[mask, 'E'].sum(),
            'Churn_Rate_%': df.loc[mask, 'E'].mean()*100,
            'Median_Survival_Months': kmf_g.median_survival_time_
        })

summary_df = pd.DataFrame(summary_data).round(1)
print(summary_df.to_string(index=False))
summary_df.to_csv('km_survival_summary.csv', index=False)

print("\n" + "=" * 65)


## KAPLAN-MEIER ANALYSIS COMPLETE


In [ ]:
print("=" * 65)
